# Combined Pipeline: Dataset Generation + Student Fine-Tuning

This single notebook generates synthetic data on-the-fly and fine-tunes sequentially,
avoiding disk bloat. Outputs LoRA adapters to HF Hub after each successful doc.

**Flow:**
1. Load teacher (70B, 4-bit).
2. For each doc: Generate in-memory → Train mini-batch → Eval → Push if better.
3. Final merge & push.

**HF Hub:** Push to `penguneedLLM/digitalbookllm-student` (set HF_TOKEN).

In [7]:
# Installs (run once)
%pip install -q transformers peft bitsandbytes accelerate datasets trl sentence-transformers faiss-cpu huggingface_hub tqdm requests pypdf2 beautifulsoup4 newspaper3k langchain-text-splitters torch

# Login to HF (use Secrets in Colab/HF Spaces)
from huggingface_hub import login
login()  # Prompts for token or uses env

Note: you may need to restart the kernel to use updated packages.


In [8]:
# Imports
import os
import json
import random
import re
import gc
import numpy as np
import faiss
import torch
from pathlib import Path
from typing import List, Tuple, Optional, Dict
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, pipeline
)
from peft import LoraConfig, PeftModel, get_peft_model
from trl import SFTTrainer
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from huggingface_hub import HfApi, Repository
from tqdm.notebook import tqdm
import requests
from bs4 import BeautifulSoup
import PyPDF2

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}, Mem: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")

PyTorch: 2.9.0+cpu, CUDA: False


In [9]:
# Config
HF_REPO_ID = "penguneedLLM/digitalbookllm-student"  # Change to your repo
TEACHER_MODEL = "meta-llama/Llama-3.1-70B-Instruct"
STUDENT_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
EMBED_MODEL = "all-MiniLM-L6-v2"
SCORE_THRESHOLD = 4.5
EXAMPLES_PER_DOC = 300  # Tune for VRAM (200-500)
TOP_K = 3
BATCH_SIZE = 2  # Adjust down if OOM
EPOCHS_PER_DOC = 1

# Quant config (4-bit for both)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=False
)

# LoRA config
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none", task_type="CAUSAL_LM"
)

# Training args (memory-efficient)
# Auto-select mixed precision: prefer bf16 when supported by CUDA/Torch,
# otherwise fall back to fp16 when a GPU is available, or full precision on CPU.
use_bf16 = torch.cuda.is_available() and getattr(torch.cuda, 'is_bf16_supported', lambda: False)()
use_fp16 = torch.cuda.is_available() and not use_bf16
print(f"Precision selection -> bf16: {use_bf16}, fp16: {use_fp16}")
training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    learning_rate=2e-4, num_train_epochs=EPOCHS_PER_DOC,
    logging_steps=10, save_strategy="no",
    output_dir="./tmp", optim="paged_adamw_32bit",
    bf16=use_bf16, fp16=use_fp16, max_grad_norm=0.3, warmup_ratio=0.03,
    group_by_length=True, report_to=None  # No wandb
)

# Doc sources (Gutenberg/Wiki/arXiv - easy URL loads)
DOC_SOURCES = [
    {"name": "the_republic", "url": "https://www.gutenberg.org/files/1497/1497-0.txt", "field": "Philosophy"},
    {"name": "leviathan", "url": "https://www.gutenberg.org/files/3207/3207-0.txt", "field": "Political Philosophy"},
    {"name": "meditations", "url": "https://www.gutenberg.org/files/2680/2680-0.txt", "field": "Stoicism"},
    {"name": "roman_republic", "url": "https://en.wikipedia.org/wiki/Roman_Republic", "field": "History", "type": "wiki"},
    # Add more; skip OpenStax/arXiv for simplicity (manual PDF upload if needed)
]

print(f"Config: Teacher={TEACHER_MODEL}, Student={STUDENT_MODEL}, Examples/Doc={EXAMPLES_PER_DOC}")
print(f"HF Repo: {HF_REPO_ID}")

Precision selection -> bf16: False, fp16: False
Config: Teacher=meta-llama/Llama-3.1-70B-Instruct, Student=mistralai/Mistral-7B-Instruct-v0.3, Examples/Doc=300
HF Repo: penguneedLLM/digitalbookllm-student


## Load Models (Teacher & Embedder)

In [ ]:
# Load teacher (once, for gen/judging)
print("Loading teacher model...")
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16
)
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL)
teacher_tokenizer.pad_token = teacher_tokenizer.eos_token

# Load embedder (lightweight)
embedder = SentenceTransformer(EMBED_MODEL)

print("Models loaded.")

Loading teacher model...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

d:\codespace\webSpace\digitalbookllm\documentation\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--meta-llama--Llama-3.1-70B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated

model.safetensors.index.json:   0%|          | 0.00/59.6k [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

...

model-00003-of-00030.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00006-of-00030.safetensors:   0%|          | 0.00/4.66G [00:00<?, ?B/s]

model-00005-of-00030.safetensors:   0%|          | 0.00/4.66G [00:00<?, ?B/s]

model-00002-of-00030.safetensors:   0%|          | 0.00/4.66G [00:00<?, ?B/s]

model-00004-of-00030.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

In [ ]:
# Doc loading helpers
def load_text_from_url(url: str, doc_type: str = "text") -> str:
    if doc_type == "wiki":
        resp = requests.get(url)
        soup = BeautifulSoup(resp.content, 'html.parser')
        text = ' '.join([p.text for p in soup.find_all('p')])
    else:  # Gutenberg TXT
        resp = requests.get(url)
        text = resp.text
        # Clean Gutenberg header/footer
        start = text.find("*** START")
        end = text.find("*** END")
        if start != -1 and end != -1:
            text = text[start+10:end].strip()
    return text

def chunk_text(text: str, chunk_size=1000, overlap=150) -> List[str]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    return splitter.split_text(text)

def build_index(chunks: List[str]) -> Tuple[faiss.Index, Dict[int, str]]:
    embeddings = embedder.encode(chunks)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings.astype('float32'))
    chunk_map = {i: chunk for i, chunk in enumerate(chunks)}
    return index, chunk_map

# RAG helpers (simplified from priority-rag.md)
def retrieve_chunks(index: faiss.Index, query: str, top_k: int = 3) -> List[str]:
    query_emb = embedder.encode([query])
    _, indices = index.search(query_emb.astype('float32'), top_k)
    return [index.chunk_map[i] for i in indices[0]]

def build_search_prompt(query: str, contexts: List[str]) -> str:
    context = "\n\n".join(contexts)
    return f"Answer based only on the context: {context}\n\nQuestion: {query}\nAnswer:"

def build_tutor_prompt(query: str, selected_text: str, related: List[str]) -> str:
    related = "\n\n".join(related)
    return f"User selected: {selected_text}\nRelated: {related}\n\nSynthesize a conversational answer to: {query}\nAnswer:"

print("Helpers loaded.")

In [ ]:
# Generation strategies (from finetune.md)
def generate_question_from_chunk(chunk: str) -> str:
    prompt = f"Generate a natural question this text answers: {chunk[:500]}...\nQuestion:"
    inputs = teacher_tokenizer(prompt, return_tensors="pt").to(teacher_model.device)
    out = teacher_model.generate(**inputs, max_new_tokens=50, temperature=0.7, do_sample=True)
    return teacher_tokenizer.decode(out[0], skip_special_tokens=True).split("Question:")[-1].strip()

def generate_search_mode_pair(index, chunk_map, all_chunks) -> Tuple[str, str]:
    chunk = random.choice(all_chunks)
    query = generate_question_from_chunk(chunk)
    contexts = retrieve_chunks(index, query, TOP_K)
    prompt = build_search_prompt(query, contexts)
    inputs = teacher_tokenizer(prompt, return_tensors="pt").to(teacher_model.device)
    out = teacher_model.generate(**inputs, max_new_tokens=200, temperature=0.2)
    completion = teacher_tokenizer.decode(out[0], skip_special_tokens=True).split("Answer:")[-1].strip()
    return prompt, completion

def generate_tutor_ask_pair(index, chunk_map, all_chunks) -> Tuple[str, str]:
    selected = random.choice(all_chunks)
    query = f"Explain this in simpler terms: {selected[:100]}..."
    augmented_query = query + " " + selected
    related = retrieve_chunks(index, augmented_query, TOP_K)
    # Dedup
    related = [r for r in related if r != selected]
    prompt = build_tutor_prompt(query, selected, related)
    # Generate completion (similar to above)
    inputs = teacher_tokenizer(prompt, return_tensors="pt").to(teacher_model.device)
    out = teacher_model.generate(**inputs, max_new_tokens=200, temperature=0.2)
    completion = teacher_tokenizer.decode(out[0], skip_special_tokens=True).split("Answer:")[-1].strip()
    return prompt, completion

def generate_tutor_query_pair(index, chunk_map, all_chunks) -> Tuple[str, str]:
    selected = random.choice(all_chunks)
    implicit_query = "Summarize this selection."
    augmented_query = implicit_query + " " + selected
    related = retrieve_chunks(index, augmented_query, TOP_K)
    related = [r for r in related if r != selected]
    prompt = build_tutor_prompt(implicit_query, selected, related)
    # Generate (similar)
    inputs = teacher_tokenizer(prompt, return_tensors="pt").to(teacher_model.device)
    out = teacher_model.generate(**inputs, max_new_tokens=200, temperature=0.2)
    completion = teacher_tokenizer.decode(out[0], skip_special_tokens=True).split("Answer:")[-1].strip()
    return prompt, completion

strategies = [generate_search_mode_pair, generate_tutor_ask_pair, generate_tutor_query_pair]

# Judging (LLM-as-judge stub; expand metrics)
def judge_response(prompt: str, completion: str) -> float:
    judge_prompt = f"""
Score this (prompt, completion) on faithfulness (0-5), relevance (0-5), clarity (0-5).
Prompt: {prompt[:500]}...
Completion: {completion[:200]}...
Scores: Faithfulness: X, Relevance: Y, Clarity: Z. Total: (X*0.4 + Y*0.3 + Z*0.3)"""
    inputs = teacher_tokenizer(judge_prompt, return_tensors="pt").to(teacher_model.device)
    out = teacher_model.generate(**inputs, max_new_tokens=50, temperature=0.1)
    total_match = re.search(r'Total:\s*([0-5\.]+)', teacher_tokenizer.decode(out[0]))
    return float(total_match.group(1)) if total_match else 0.0

print("Generation/Judging functions ready.")

In [ ]:
# Student loading/eval helpers
def load_student(current_path: str) -> Tuple[PeftModel, AutoTokenizer]:
    if current_path == STUDENT_MODEL:  # Base
        model = AutoModelForCausalLM.from_pretrained(current_path, quantization_config=bnb_config, device_map="auto")
        model = get_peft_model(model, lora_config)
    else:  # From HF repo
        model = PeftModel.from_pretrained(STUDENT_MODEL, current_path, device_map="auto")
        model = model.merge_and_unload()  # Temp merge for eval
        model = get_peft_model(model, lora_config)  # New adapters for training
    tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
    tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer

# Stub eval (replace with real test set)
def evaluate_model(model, tokenizer, test_prompts: List[str] = None) -> float:
    if test_prompts is None:
        test_prompts = ["Explain quantum mechanics simply."] * 5  # Dummy
    scores = []
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto")
    for p in test_prompts:
        out = pipe(p, max_new_tokens=100, do_sample=False)[0]['generated_text']
        scores.append(len(out) / 100)  # Dummy score; use real metric
    return np.mean(scores)

# HF push helper
def push_lora_to_hub(repo_id: str, local_dir: str, bitmask: int, doc_idx: int):
    api = HfApi()
    repo = Repository(local_dir, clone_from=repo_id, skip_readonly=True)
    commit_msg = f"Update LoRA: bitmask={bitmask:04b}, doc={doc_idx}"
    repo.git_add()
    repo.git_commit(commit_msg)
    repo.git_push()
    repo_path = f"{repo_id}/adapters_bitmask_{bitmask:04b}"
    api.create_repo(repo_path, exist_ok=True)
    model.save_pretrained(repo_path)
    tokenizer.save_pretrained(repo_path)
    print(f"Pushed to {repo_path}")
    return repo_path

print("Student helpers ready.")

## Main Loop: Per-Document Generation + Training

In [ ]:
# Format for SFT
def format_for_sft(pairs: List[Dict]) -> Dataset:
    texts = [f"<s>[INST] {p['prompt']} [/INST] {p['completion']} </s>" for p in pairs]
    return Dataset.from_dict({"text": texts})

# Loop
best_path = STUDENT_MODEL  # Start with base
best_score = 0.0
trained_bitmask = 0
num_docs = len(DOC_SOURCES)

# Initial eval
temp_model, temp_tok = load_student(best_path)
best_score = evaluate_model(temp_model, temp_tok)
del temp_model, temp_tok
gc.collect(); torch.cuda.empty_cache()
print(f"Base score: {best_score:.3f}")

for i, source in enumerate(DOC_SOURCES):
    print(f"\n{'='*60}")
    print(f"Doc {i+1}/{num_docs}: {source['name']} ({source['field']})")
    print(f"{'='*60}")
    
    # 1. Load doc
    text = load_text_from_url(source['url'], source.get('type', 'text'))
    chunks = chunk_text(text)
    index, chunk_map = build_index(chunks)
    
    # 2. Generate in-memory
    high_quality = []
    pbar = tqdm(range(EXAMPLES_PER_DOC), desc="Gen")
    for _ in pbar:
        strat = random.choice(strategies)
        prompt, comp = strat(index, chunk_map, chunks)
        if prompt and comp:
            score = judge_response(prompt, comp)
            pbar.set_postfix(score=f"{score:.2f}")
            if score >= SCORE_THRESHOLD:
                high_quality.append({"prompt": prompt, "completion": comp})
    print(f"Generated {len(high_quality)} high-quality pairs.")
    
    if len(high_quality) < 10:  # Skip small
        continue
    
    # 3. Train
    ds = format_for_sft(high_quality)
    model, tokenizer = load_student(best_path)
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=ds,
        peft_config=lora_config, dataset_text_field="text",
        max_seq_length=2048, args=training_args
    )
    trainer.train()
    
    # 4. Temp save & eval
    temp_dir = f"./temp_lora_{i}"
    trainer.model.save_pretrained(temp_dir)
    model = model.merge_and_unload()
    new_score = evaluate_model(model, tokenizer)
    print(f"New score: {new_score:.3f} (vs best {best_score:.3f})")
    
    # 5. Rollback/Push
    current_bit = 1 << i
    new_mask = trained_bitmask | current_bit
    if new_score > best_score:
        best_path = push_lora_to_hub(HF_REPO_ID, temp_dir, new_mask, i)
        best_score = new_score
        trained_bitmask = new_mask
        print("✓ Improvement! Updated HF repo.")
    else:
        print("✗ No improvement. Rolled back.")
    
    # 6. Cleanup
    del model, tokenizer, trainer, ds
    gc.collect(); torch.cuda.empty_cache()
    Path(temp_dir).unlink(missing_ok=True)  # Quick delete

print(f"\nFinal: Best path={best_path}, Bitmask={trained_bitmask:0{num_docs}b}, Score={best_score:.3f}")

In [ ]:
# Final merge & push (optional, for inference-ready model)
print("Merging final model...")
base = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL, device_map="auto", torch_dtype=torch.float16)
final_model = PeftModel.from_pretrained(base, best_path)
final_model = final_model.merge_and_unload()
final_path = f"{HF_REPO_ID}/final_merged"
final_model.push_to_hub(final_path)
AutoTokenizer.from_pretrained(STUDENT_MODEL).push_to_hub(final_path)
print(f"Pushed merged model to {final_path}")

# Cleanup teacher
del teacher_model
gc.collect(); torch.cuda.empty_cache()

## Next Steps
- **Test Inference**: Load from `penguneedLLM/digitalbookllm-student/final_merged`.
- **Expand Eval**: Upload a test dataset to HF Datasets for real metrics.
- **Scale Docs**: Add PDF loaders for OpenStax (upload files to Colab).
- **Monitor**: Check HF repo commits for bitmask tracking.

This pipeline uses <10GB disk (mostly temp), pushes ~1-2GB to HF per update.